# 03 - Train a classifier and export it for the FPGA

This notebook trains the network the paper deploys and converts it to HLS C++ with hls4ml, from the recorded dataset to
an hls4ml project you can synthesize (paper Sec. VI).

The network is the one family of Di Guglielmo et al.: **N inputs → 4 hidden units → 1 output**. Both dense layers have
ternary weights {−1, 0, +1} (QKeras), with batch normalization and ReLU between them and a sigmoid at the end. The input is
a window of the readout trace, interleaved I and Q values.

Three things make the trained network match the hardware:
1. **Domain map.** The network runs on what the ADC delivers after the loopback, not on the recording. A label-free affine
   map $z \mapsto k_2 z + c_2$, fit once on 600 replayed shots, moves the training data into that domain. For the low-latency
   *boxcar* readout path, the map also includes the path's moving sum and a sub-sample shift.
2. **Signed integer input.** The firmware reads each ADC word as a signed 16-bit integer; training applies the same
   rounding and clipping.
3. **Seed selection.** Several seeds are trained and the one with the lowest validation loss is kept. Validation is a 20%
   split of the training rows, so no test or replayed shot is involved.

Then hls4ml converts the network with the precisions the firmware uses.

**The deployed networks** (paper Table II), and the settings that reproduce them:

| Network | `PATH` | `WINDOW` | `ROWS` |
|---|---|---|---|
| NN_live, boxcar path (deployed) | `"box"` | 576 | `0:10000,110000:300000` |
| NN_live, FIR path | `"fir"` | 576 | `0:10000,110000:300000` |
| NN_axi | `"fir"` | 770 | `0:10000,110000:300000` |

Rows 10,000–110,000 are skipped because the synthetic-data generator (notebook 06) was fit on them.

**You need:** the dataset's training split (`0528_X_train_0_770.npy`, `0528_y_train_0_770.npy`), the 20k test subset
(`data/test20k_real_X.npy`, `data/test20k_real_y.npy`), and one replayed capture of that subset: `data/domain_calib_raw_X.npy` /
`data/domain_calib_acquired.npy` for the FIR path, or a boxcar capture (`loopback20k_box_A.npz` + `.json`) for the boxcar path.
All are in `DATA_MANIFEST.csv`. Python packages: TensorFlow, QKeras, hls4ml.

In [ ]:
from pathlib import Path

DATA_DIR = Path("..")
OUT_DIR = Path("outputs/training")

PATH = "box"                         # "box" (boxcar readout path, deployed NN_live) or "fir" (QICK's FIR path)
WINDOW = 576                         # samples of the trace the network sees, from the start
ROWS = "0:10000,110000:300000"       # training rows of 0528_X_train_0_770
SEEDS = [20260901, 20260902, 20260903, 20260904, 20260905]
EPOCHS, BATCH, PATIENCE = 50, 32, 5

# domain-map inputs
FIR_RAW, FIR_ACQ = "data/domain_calib_raw_X.npy", "data/domain_calib_acquired.npy"   # test20k rows 1000:1600 and their replay
BOX_CAPTURE = "loopback20k_box_A"                                           # .npz (rows, box, fir) and .json (calibration c)
BOX_M = 11                                                                  # the boxcar's moving-sum length
BOX_MAP_JSON = "calibration/box_domain_map.json"     # the map the deployed NN_live used; set to None to refit it from BOX_CAPTURE
CALIB_ROWS = (1000, 1600)                                                   # test20k rows used for the map, never scored

FPGA_PART, CLOCK_NS = "xczu48dr-ffvg1517-2-e", 10.0
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_USE_LEGACY_KERAS"] = "1"        # QKeras needs the legacy Keras API; set before importing TensorFlow
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import json

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from qkeras.qlayers import QDense
from qkeras.quantizers import ternary
from tensorflow.keras.layers import Activation, BatchNormalization, Input


def cz(X):
    X = np.asarray(X, dtype=np.float64)
    return X[..., 0::2] + 1j * X[..., 1::2]


def interleave(z):
    out = np.empty(z.shape[:-1] + (2 * z.shape[-1],), dtype=np.float32)
    out[..., 0::2], out[..., 1::2] = z.real, z.imag
    return out

## 1. Domain map

**FIR path.** Complex least squares between the *mean* trajectory of 600 recorded test traces and the mean of their
replay: $\bar z_\text{acq} = k_2 \bar z_\text{rec} + c_2$. Using the mean over all shots, with no labels, means the map
carries no class information.

**Boxcar path.** Readout 0's boxcar path averages each group of 8 ADC samples and then takes an $M$-sample moving sum at
the decimated rate. Its map is $\text{box}[n] \approx k_b\,\text{MA}_M(z)[n-s] + c_b$, with the moving average primed with
the DAC's silence level (the calibration offset $c$) and the shift $s$ found on a quarter-sample grid.

In [ ]:
def fit_affine(zr_mean, za_mean):
    A = np.vstack([zr_mean, np.ones_like(zr_mean)]).T
    (k, c), *_ = np.linalg.lstsq(A, za_mean, rcond=None)
    return complex(k), complex(c)


def moving_average(z, M, prime):
    # Causal M-sample moving average along the last axis, primed with `prime` before n = 0.
    pad = np.concatenate([np.full(z.shape[:-1] + (M - 1,), prime, dtype=z.dtype), z], axis=-1)
    cs = np.concatenate([np.zeros(z.shape[:-1] + (1,), dtype=z.dtype), np.cumsum(pad, axis=-1)], axis=-1)
    return (cs[..., M:] - cs[..., :-M]) / M


def shift(z, s):
    # z[n - s] by linear interpolation, edges held.
    n = np.arange(z.shape[-1], dtype=float)
    src = np.clip(n - s, 0, z.shape[-1] - 1)
    i0 = np.floor(src).astype(int); i1 = np.minimum(i0 + 1, z.shape[-1] - 1); f = src - i0
    return z[..., i0] * (1 - f) + z[..., i1] * f


def fit_box_map(zr_mean, za_mean, M, prime, lo=8, hi=760, shifts=np.arange(-12, 12.01, 0.25)):
    best = None
    f = moving_average(zr_mean[None], M, prime)[0]
    for s in shifts:
        g = shift(f, s)[lo:hi]
        k, c = fit_affine(g, za_mean[lo:hi])
        e = float(np.sqrt(np.mean(np.abs(za_mean[lo:hi] - (k * g + c)) ** 2)))
        if best is None or e < best["rms"]:
            best = dict(rms=e, s=float(s), k=k, c=c)
    return best


if PATH == "fir":
    zr, za = cz(np.load(DATA_DIR / FIR_RAW)), cz(np.load(DATA_DIR / FIR_ACQ))
    k2, c2 = fit_affine(zr.mean(0), za.mean(0))
    domain = dict(path="fir", k=k2, c=c2)
    apply_map = lambda z: k2 * z + c2
    print(f"FIR map: k2 = {k2:.5f} (|k2| {abs(k2):.4f}, arg {np.degrees(np.angle(k2)):+.2f} deg), c2 = {c2:.3f}")
else:
    cap = np.load(DATA_DIR / f"{BOX_CAPTURE}.npz")
    prime = complex(*json.load(open(DATA_DIR / f"{BOX_CAPTURE}.json"))["c"])    # DAC silence = calibration offset c
    ref = np.load(DATA_DIR / "data/test20k_real_X.npy", mmap_mode="r")
    sel = (cap["rows"] >= CALIB_ROWS[0]) & (cap["rows"] < CALIB_ROWS[1])
    zr, za = cz(ref[cap["rows"][sel]]), cz(cap["box"][sel])
    if BOX_MAP_JSON and (DATA_DIR / BOX_MAP_JSON).exists():          # reuse the deployed map exactly
        j = json.load(open(DATA_DIR / BOX_MAP_JSON))
        BOX_M, prime = j["m"], complex(*j["prime"])
        b = dict(s=j["shift"], k=complex(*j["k"]), c=complex(*j["c"]), rms=float("nan"))
        print("using the saved boxcar map", BOX_MAP_JSON)
    else:
        b = fit_box_map(zr.mean(0), za.mean(0), BOX_M, prime)
    domain = dict(path="box", m=BOX_M, shift=b["s"], k=b["k"], c=b["c"], prime=prime)
    apply_map = lambda z: b["k"] * shift(moving_average(z, BOX_M, prime), b["s"]) + b["c"]
    print(f"boxcar map on {sel.sum()} shots: s = {b['s']:+.2f} samples, |k| = {abs(b['k']):.4f}, "
          f"arg {np.degrees(np.angle(b['k'])):+.1f} deg, rms residual {b['rms']:.2f}")

pred = apply_map(zr).mean(0)
t = np.arange(za.shape[1]) * 1.808e-3
plt.figure(figsize=(8, 2.8))
plt.plot(t, np.abs(za.mean(0)), label="replayed (mean)")
plt.plot(t, np.abs(pred), "--", label="map(recorded) (mean)")
plt.xlabel("time (us)"); plt.ylabel("|z|"); plt.legend(); plt.tight_layout(); plt.show()

## 2. Training data

Map the training rows into the hardware domain, then apply what the firmware does to each ADC word: round, clip to the
signed 16-bit range. Keep the first `WINDOW` samples (`2*WINDOW` inputs).

In [ ]:
def hw_input(X, window):
    Xi = np.clip(np.round(X), -32768, 32767).astype(np.int16).astype(np.float32)
    return Xi[:, :2 * window]


ranges = [tuple(int(v) for v in r.split(":")) for r in ROWS.split(",")]
Xsrc = np.load(DATA_DIR / "0528_X_train_0_770.npy", mmap_mode="r")
ysrc = np.load(DATA_DIR / "0528_y_train_0_770.npy")
parts = []
for lo, hi in ranges:
    for b0 in range(lo, hi, 20000):
        b1 = min(hi, b0 + 20000)
        parts.append(hw_input(interleave(apply_map(cz(Xsrc[b0:b1]))), WINDOW))
X_train = np.concatenate(parts)
y_train = np.concatenate([ysrc[lo:hi] for lo, hi in ranges]).astype(np.float32)
print(f"training set {X_train.shape}, balance {np.bincount(y_train.astype(int))}, "
      f"input range [{X_train.min():.0f}, {X_train.max():.0f}]")

## 3. Train

Adam (learning rate $10^{-3}$), binary cross-entropy, batch 32, up to 50 epochs with early stopping on validation loss
(patience 5). One model per seed; keep the lowest validation loss.

In [ ]:
def build_model(n_inputs):
    L2 = tf.keras.regularizers.l2(1e-3)
    m = tf.keras.Sequential([
        Input(shape=(n_inputs,), name="input_1"),
        QDense(4, kernel_quantizer=ternary(), bias_quantizer=ternary(), name="q_dense_1"),
        BatchNormalization(name="batchnorm1", gamma_regularizer=L2, beta_regularizer=L2),
        Activation("relu", name="relu1"),
        QDense(1, kernel_quantizer=ternary(), bias_quantizer=ternary(), kernel_regularizer=L2,
               bias_regularizer=L2, name="q_output"),
        Activation("sigmoid", name="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m


best = None
for seed in SEEDS:
    tf.keras.utils.set_random_seed(seed)
    perm = np.random.default_rng(seed).permutation(len(y_train))
    m = build_model(X_train.shape[1])
    h = m.fit(X_train[perm], y_train[perm], validation_split=0.2, epochs=EPOCHS, batch_size=BATCH, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)])
    vl = float(min(h.history["val_loss"]))
    print(f"seed {seed}: val_loss {vl:.5f}, val_acc {100 * h.history['val_accuracy'][int(np.argmin(h.history['val_loss']))]:.2f}%, "
          f"{len(h.history['val_loss'])} epochs")
    if best is None or vl < best[0]:
        best = (vl, seed, m, h.history)
val_loss, seed_used, model, hist = best
print(f"selected seed {seed_used} (val_loss {val_loss:.5f})")
model.save(OUT_DIR / f"model_{PATH}_w{WINDOW}.h5")

## 4. Quick check on held-out test traces

Score the selected network (floating point) on 2,000 test traces mapped into the same domain. The calibration rows are
excluded. This is a sanity check; the paper's fidelities come from replayed traces and the bit-accurate model
(notebook 04).

In [ ]:
Xt_all = np.load(DATA_DIR / "data/test20k_real_X.npy", mmap_mode="r")
yt_all = np.load(DATA_DIR / "data/test20k_real_y.npy").astype(int)
keep = np.r_[0:CALIB_ROWS[0], CALIB_ROWS[1]:3000][:2000]
Xt = hw_input(interleave(apply_map(cz(Xt_all[keep]))), WINDOW)
yt = yt_all[keep]
p = model.predict(Xt, verbose=0).reshape(-1)
pred = (p > 0.5).astype(int)
p10, p01 = np.mean(pred[yt == 0] == 1), np.mean(pred[yt == 1] == 0)
print(f"fidelity {100 * (1 - (p10 + p01) / 2):.2f}%   P(1|0) {100 * p10:.2f}%   P(0|1) {100 * p01:.2f}%   (n = {len(yt)})")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(hist["loss"], label="train"); ax[0].plot(hist["val_loss"], label="validation")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
ax[1].hist(p[yt == 0], 50, alpha=0.6, label="|0⟩"); ax[1].hist(p[yt == 1], 50, alpha=0.6, label="|1⟩")
ax[1].set_xlabel("network output"); ax[1].legend(); plt.tight_layout(); plt.show()

## 5. Export with hls4ml

The precisions below are the firmware's. The input is `ap_fixed<32,20>`: a signed 16-bit sample fits with room to spare.
The first layer accumulates in `ap_fixed<48,32>`, batch normalization uses `ap_fixed<40,8>` scale and bias, and the
sigmoid output is `ap_fixed<18,8>`. Fully parallel I/O, Resource strategy, reuse factor 1, a 10 ns HLS clock target. The
result is a Vivado HLS project in `OUT_DIR`. Our streaming NN_live core is hand-written HLS around these weights
(`neural_net4_live576s/`); NN_axi uses this network as generated, inside its AXI-Stream wrapper (`neural_net4_tlast/`).

In [ ]:
import hls4ml


def make_hls_config(model):
    cfg = hls4ml.utils.config_from_keras_model(model, granularity="name")
    cfg["Model"].update(Precision="ap_fixed<16,6>", ReuseFactor=1, Strategy="Resource")
    L = cfg["LayerName"]

    def prec(layer):
        if not isinstance(L[layer].get("Precision"), dict):
            L[layer]["Precision"] = {}
        return L[layer]["Precision"]

    def result(layer, p):
        if layer in L:
            prec(layer)["result"] = p
            L[layer]["Trace"] = True

    for name in ("input_1", "input_1_input", "q_dense_1_input"):
        if name in L:
            L[name]["Precision"] = "ap_fixed<32,20>"
    result("q_dense_1", "ap_fixed<48,32>"); L["q_dense_1"]["accum_t"] = "ap_fixed<48,32>"
    result("q_output", "ap_fixed<32,16>"); L["q_output"]["accum_t"] = "ap_fixed<32,16>"
    for name in ("batchnorm1", "relu1"):
        result(name, "ap_fixed<32,16>")
    result("sigmoid", "ap_fixed<18,8>")
    for name in list(L):
        if "alpha" in name or "linear" in name:
            result(name, "ap_fixed<24,10>")
    result("q_dense_1_linear", "ap_fixed<48,32>")
    result("q_output_linear", "ap_fixed<32,16>")
    if "batchnorm1" in L:
        prec("batchnorm1")["scale"] = "ap_fixed<40,8>"
        prec("batchnorm1")["bias"] = "ap_fixed<40,8>"
    return cfg


hls_cfg = make_hls_config(model)
prj = OUT_DIR / f"hls4ml_{PATH}_w{WINDOW}"
hls_model = hls4ml.converters.convert_from_keras_model(model=model, hls_config=hls_cfg, output_dir=str(prj),
                                                       part=FPGA_PART, io_type="io_parallel", clock_period=CLOCK_NS,
                                                       backend="Vivado", project_name="NN")
hls_model.write()
json.dump(dict(path=PATH, window=WINDOW, rows=ROWS, seed=seed_used, val_loss=val_loss,
               domain={k: (str(v) if isinstance(v, complex) else v) for k, v in domain.items()}),
          open(OUT_DIR / f"model_{PATH}_w{WINDOW}.json", "w"), indent=1)
print("hls4ml project:", prj, "| weights:", sorted(p.name for p in (prj / "firmware" / "weights").glob("*"))[:6])

**Next:** notebook 04 checks the fixed-point network against floating point on replayed traces, which is the check that
the hardware will make the same decisions as the trained model.

**Things to try:** a shorter `WINDOW` (paper Fig. 5 shows the fidelity cost per nanosecond saved), more hidden units in
`build_model`, or training on your own qubit's recorded traces after replaying them through notebook 02.